# E-Commerce Customer Churn Prediction
**Goal:** Predict whether a customer will churn based on demographics, activity, purchase behaviour, and engagement patterns.

**Workflow:**
1. Imports
2. Data Loading & Initial Inspection
3. Data Cleaning
4. Outlier Handling
5. Exploratory Data Analysis (EDA)
6. Encoding & Train/Test Split
7. Scaling
8. Model Comparison — Default Parameters
9. Model Comparison — SMOTE Oversampling (Default Params)
10. Model Comparison — Random Undersampling (Default Params)
11. Default Summary Table — All 3 Strategies
12. Hyperparameter Tuning — Top Models on Original Data
13. Hyperparameter Tuning — Top Models on SMOTE Data
14. Hyperparameter Tuning — Top Models on Undersampled Data
15. Grand Summary Table — All 6 Strategies
16. Final Model Evaluation & Threshold Tuning
17. Overfitting / Underfitting Check
18. Feature Importances
19. Save Artifacts

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix, roc_curve)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

sns.set_theme(style='whitegrid', palette='muted')

## 2. Data Loading & Initial Inspection

In [ ]:
df = pd.read_csv('ecommerce_customer_churn_dataset.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Data Cleaning

| Column | Reason for Dropping |
|---|---|
| `Country` | High cardinality — too many unique values, noisy |
| `City` | Same as Country — too granular |
| `Signup_Quarter` | Very weak signal for churn prediction |

In [ ]:
df = df.drop(columns=['Country', 'City', 'Signup_Quarter'])
print("Shape after dropping columns:", df.shape)

In [ ]:
print("Null counts:")
print(df.isnull().sum())

In [ ]:
df.fillna(df.median(numeric_only=True), inplace=True)
print("Nulls after filling:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())

In [ ]:
int_columns = ['Age','Wishlist_Items','Total_Purchases','Days_Since_Last_Purchase',
               'Customer_Service_Calls','Product_Reviews_Written','Payment_Method_Diversity']
df[int_columns] = df[int_columns].astype(int)
df['Gender'] = df['Gender'].astype('category')
print("Data types fixed.")

## 4. Outlier Handling
- **Age**: Filtered to 15–90 (removed values like 5, 150, 200)
- **Cart_Abandonment_Rate**: Clipped at 100 (percentage — cannot exceed 100)

Outliers properly managed → **StandardScaler** is appropriate.

In [ ]:
print("Age extremes:", sorted(df['Age'].unique())[:5], '...', sorted(df['Age'].unique())[-5:])
print("Cart_Abandonment_Rate max:", df['Cart_Abandonment_Rate'].max())

In [ ]:
df = df[(df['Age'] >= 15) & (df['Age'] <= 90)]
df['Cart_Abandonment_Rate'] = df['Cart_Abandonment_Rate'].clip(upper=100)
print("Shape after outlier handling:", df.shape)

## 5. Exploratory Data Analysis (EDA)

### 5.1 Target Variable — Churn Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
churn_counts = df['Churned'].value_counts()
axes[0].bar(['Not Churned (0)', 'Churned (1)'], churn_counts.values,
            color=['steelblue', 'coral'], edgecolor='white')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')
axes[0].set_title('Churn Count')
axes[1].pie(churn_counts.values, labels=['Not Churned', 'Churned'],
            autopct='%1.1f%%', colors=['steelblue', 'coral'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Percentage')
plt.suptitle('Target Variable: Customer Churn Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2 Demographics vs Churn

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(data=df, x='Age', hue='Churned', bins=30, kde=True,
             palette=['steelblue', 'coral'], ax=axes[0,0])
axes[0,0].set_title('Age Distribution by Churn')
gender_churn = df.groupby(['Gender', 'Churned']).size().unstack()
gender_churn.plot(kind='bar', ax=axes[0,1], color=['steelblue', 'coral'], edgecolor='white', rot=0)
axes[0,1].set_title('Gender vs Churn')
axes[0,1].legend(['Not Churned', 'Churned'])
sns.boxplot(x='Churned', y='Membership_Years', data=df,
            palette=['steelblue', 'coral'], ax=axes[1,0])
axes[1,0].set_title('Membership Years vs Churn')
axes[1,0].set_xticklabels(['Not Churned', 'Churned'])
sns.histplot(data=df, x='Login_Frequency', hue='Churned', bins=30, kde=True,
             palette=['steelblue', 'coral'], ax=axes[1,1])
axes[1,1].set_title('Login Frequency by Churn')
plt.suptitle('Demographics vs Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.3 Activity, Purchase & Engagement vs Churn

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
features = [
    ('Session_Duration_Avg',   'Session Duration',      'hist'),
    ('Cart_Abandonment_Rate',  'Cart Abandonment Rate', 'box'),
    ('Customer_Service_Calls', 'Service Calls',         'box'),
    ('Discount_Usage_Rate',    'Discount Usage Rate',   'hist'),
    ('Email_Open_Rate',        'Email Open Rate',       'box'),
    ('Lifetime_Value',         'Lifetime Value',        'hist'),
]
for ax, (col, title, kind) in zip(axes.flatten(), features):
    if kind == 'box':
        sns.boxplot(x='Churned', y=col, data=df, palette=['steelblue','coral'], ax=ax)
        ax.set_xticklabels(['Not Churned', 'Churned'])
        ax.set_xlabel('')
    else:
        sns.histplot(data=df, x=col, hue='Churned', bins=30, kde=True,
                     palette=['steelblue','coral'], ax=ax)
    ax.set_title(title)
plt.suptitle('Activity, Purchase & Engagement vs Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.4 Correlation Heatmap & Feature Correlation with Churn

In [ ]:
plt.figure(figsize=(14, 10))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, square=True, annot_kws={'size': 7})
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
churn_corr = df.corr(numeric_only=True)['Churned'].drop('Churned').sort_values(ascending=False)
plt.figure(figsize=(10, 6))
colors = ['coral' if v > 0 else 'steelblue' for v in churn_corr]
churn_corr.plot(kind='barh', color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlations with Churn', fontsize=12, fontweight='bold')
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()
print(churn_corr)

## 6. Encoding & Train/Test Split

In [ ]:
X = df.drop('Churned', axis=1)
y = df['Churned']
X = pd.get_dummies(X, drop_first=True)
print("Feature shape:", X.shape)
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Churn rate — Train: {y_train.mean():.3f}  |  Test: {y_test.mean():.3f}")

## 7. Scaling — StandardScaler

Outliers properly managed → StandardScaler is appropriate.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## 8. Model Comparison — Default Parameters

All 9 models with default settings on original data. Establishes baseline.

In [ ]:
# ── Helper: evaluate all models ──
def evaluate_models(X_tr, X_te, y_tr, y_te, label):
    models = {
        "Logistic Regression" : LogisticRegression(max_iter=1000),
        "Decision Tree"       : DecisionTreeClassifier(),
        "Random Forest"       : RandomForestClassifier(),
        "Gradient Boosting"   : GradientBoostingClassifier(),
        "AdaBoost"            : AdaBoostClassifier(),
        "Naive Bayes"         : GaussianNB(),
        "SVM"                 : SVC(probability=True),
        "KNN"                 : KNeighborsClassifier(),
        "XGBoost"             : XGBClassifier(eval_metric='logloss')
    }
    rows = []
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        y_pred_m = model.predict(X_te)
        y_prob_m = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None
        rows.append({
            "Model"    : name,
            "Accuracy" : round(accuracy_score(y_te, y_pred_m), 4),
            "Precision": round(precision_score(y_te, y_pred_m, zero_division=0), 4),
            "Recall"   : round(recall_score(y_te, y_pred_m), 4),
            "F1"       : round(f1_score(y_te, y_pred_m), 4),
            "ROC-AUC"  : round(roc_auc_score(y_te, y_prob_m), 4) if y_prob_m is not None else None,
        })
    df_res = pd.DataFrame(rows).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
    df_res.columns = ["Model"] + [f"{c} ({label})" for c in df_res.columns[1:]]
    return df_res

default_results = evaluate_models(X_train_scaled, X_test_scaled, y_train, y_test, "Default")
default_results

## 9. Model Comparison — SMOTE Oversampling (Default Params)

Synthetically balance minority class → retrain all models with default params.

In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print("Before SMOTE:", y_train.value_counts().to_dict())
print("After  SMOTE:", pd.Series(y_train_sm).value_counts().to_dict())

scaler_sm = StandardScaler()
X_train_sm_scaled = scaler_sm.fit_transform(X_train_sm)
X_test_sm_scaled  = scaler_sm.transform(X_test)

smote_default_results = evaluate_models(X_train_sm_scaled, X_test_sm_scaled,
                                         y_train_sm, y_test, "SMOTE-Default")
smote_default_results

## 10. Model Comparison — Random Undersampling (Default Params)

Reduce majority class to match minority → retrain all models with default params.

In [ ]:
rus = RandomUnderSampler(random_state=42)
X_train_us, y_train_us = rus.fit_resample(X_train, y_train)
print("Before Undersampling:", y_train.value_counts().to_dict())
print("After  Undersampling:", pd.Series(y_train_us).value_counts().to_dict())

scaler_us = StandardScaler()
X_train_us_scaled = scaler_us.fit_transform(X_train_us)
X_test_us_scaled  = scaler_us.transform(X_test)

under_default_results = evaluate_models(X_train_us_scaled, X_test_us_scaled,
                                         y_train_us, y_test, "Under-Default")
under_default_results

## 11. Default Summary Table — All 3 Strategies

Compare all models across Default / SMOTE / Undersampling with default parameters.
**Pick top 3 models by ROC-AUC → tune only those in next sections.**

In [ ]:
default_summary = default_results.merge(smote_default_results, on="Model")                                    .merge(under_default_results, on="Model")
print("=== Default Params Summary ===")
default_summary

In [ ]:
# Visual comparison
roc_cols = [c for c in default_summary.columns if 'ROC-AUC' in c]
default_summary[['Model'] + roc_cols].set_index('Model').plot(
    kind='barh', figsize=(13, 6), colormap='Set2', edgecolor='white')
plt.title('ROC-AUC — Default vs SMOTE-Default vs Under-Default',
          fontsize=13, fontweight='bold')
plt.xlabel('ROC-AUC')
plt.axvline(0.5, color='red', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Pick top 3 models by best ROC-AUC across any default strategy ──
roc_cols = [c for c in default_summary.columns if 'ROC-AUC' in c]
default_summary['Best_Default_ROC'] = default_summary[roc_cols].max(axis=1)
top3_models = default_summary.nlargest(3, 'Best_Default_ROC')['Model'].tolist()
print("Top 3 models selected for hyperparameter tuning:")
for i, m in enumerate(top3_models, 1):
    print(f"  {i}. {m}")

## 12. Hyperparameter Tuning — Top Models on Original Data

GridSearchCV on top 3 models only — original (unsampled) data.
`cv=3` and `n_jobs=-1` for speed.

In [ ]:
# ── Compact param grids ──
all_param_grids = {
    "Logistic Regression" : {"C": [0.1, 1, 10]},
    "Decision Tree"       : {"max_depth": [3, 5, 7], "criterion": ["gini", "entropy"]},
    "Random Forest"       : {"n_estimators": [100, 200], "max_depth": [5, 7]},
    "Gradient Boosting"   : {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [3, 5]},
    "AdaBoost"            : {"n_estimators": [100, 200], "learning_rate": [0.1, 1.0]},
    "Naive Bayes"         : {},
    "SVM"                 : {"C": [0.1, 1, 10], "kernel": ["rbf", "linear"]},
    "KNN"                 : {"n_neighbors": [3, 5, 7], "weights": ["uniform", "distance"]},
    "XGBoost"             : {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [3, 5]},
}

all_model_instances = {
    "Logistic Regression" : LogisticRegression(max_iter=1000),
    "Decision Tree"       : DecisionTreeClassifier(),
    "Random Forest"       : RandomForestClassifier(),
    "Gradient Boosting"   : GradientBoostingClassifier(random_state=42),
    "AdaBoost"            : AdaBoostClassifier(random_state=42),
    "Naive Bayes"         : GaussianNB(),
    "SVM"                 : SVC(probability=True),
    "KNN"                 : KNeighborsClassifier(),
    "XGBoost"             : XGBClassifier(eval_metric='logloss', random_state=42),
}

def tune_top_models(X_tr, X_te, y_tr, y_te, label, top_models):
    """Run GridSearchCV only on top_models list."""
    tuned_store = {}
    rows = []
    for name in top_models:
        model = all_model_instances[name]
        pg    = all_param_grids[name]
        if pg:
            gs = GridSearchCV(model, pg, cv=3, scoring='roc_auc', n_jobs=-1, verbose=0)
            gs.fit(X_tr, y_tr)
            best = gs.best_estimator_
            print(f"{name:25s} | Best: {gs.best_params_}")
        else:
            best = model
            best.fit(X_tr, y_tr)
            print(f"{name:25s} | No params to tune")
        tuned_store[name] = best
        y_pred_t = best.predict(X_te)
        y_prob_t = best.predict_proba(X_te)[:, 1] if hasattr(best, 'predict_proba') else None
        rows.append({
            "Model"    : name,
            "Accuracy" : round(accuracy_score(y_te, y_pred_t), 4),
            "Precision": round(precision_score(y_te, y_pred_t, zero_division=0), 4),
            "Recall"   : round(recall_score(y_te, y_pred_t), 4),
            "F1"       : round(f1_score(y_te, y_pred_t), 4),
            "ROC-AUC"  : round(roc_auc_score(y_te, y_prob_t), 4) if y_prob_t is not None else None,
        })
    df_res = pd.DataFrame(rows).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
    df_res.columns = ["Model"] + [f"{c} ({label})" for c in df_res.columns[1:]]
    return df_res, tuned_store

print("=== Tuning Top 3 on Original Data ===")
tuned_orig_results, tuned_orig_store = tune_top_models(
    X_train_scaled, X_test_scaled, y_train, y_test, "Tuned-Original", top3_models)
tuned_orig_results

## 13. Hyperparameter Tuning — Top Models on SMOTE Data

Same top 3 models, now tuned on SMOTE-resampled data.

In [ ]:
print("=== Tuning Top 3 on SMOTE Data ===")
tuned_smote_results, tuned_smote_store = tune_top_models(
    X_train_sm_scaled, X_test_sm_scaled, y_train_sm, y_test, "Tuned-SMOTE", top3_models)
tuned_smote_results

## 14. Hyperparameter Tuning — Top Models on Undersampled Data

Same top 3 models, now tuned on undersampled data.

In [ ]:
print("=== Tuning Top 3 on Undersampled Data ===")
tuned_under_results, tuned_under_store = tune_top_models(
    X_train_us_scaled, X_test_us_scaled, y_train_us, y_test, "Tuned-Under", top3_models)
tuned_under_results

## 15. Grand Summary Table — All 6 Strategies

Mirrors mentor's approach exactly:
- Default · SMOTE-Default · Under-Default · Tuned-Original · Tuned-SMOTE · Tuned-Under

Best model & strategy auto-selected by highest ROC-AUC.

In [ ]:
# Merge all 6 strategies (only top 3 models have tuned rows)
grand_summary = default_summary[['Model'] + roc_cols]    .merge(tuned_orig_results[['Model', [c for c in tuned_orig_results.columns if 'ROC-AUC' in c][0]]], on='Model', how='left')    .merge(tuned_smote_results[['Model', [c for c in tuned_smote_results.columns if 'ROC-AUC' in c][0]]], on='Model', how='left')    .merge(tuned_under_results[['Model', [c for c in tuned_under_results.columns if 'ROC-AUC' in c][0]]], on='Model', how='left')

print("=== Grand Summary — ROC-AUC across all 6 strategies ===")
grand_summary

In [ ]:
# Visual
all_roc_cols = [c for c in grand_summary.columns if 'ROC-AUC' in c]
grand_summary[['Model'] + all_roc_cols].set_index('Model').plot(
    kind='barh', figsize=(16, 8), colormap='tab10', edgecolor='white')
plt.title('ROC-AUC — All 6 Strategies', fontsize=13, fontweight='bold')
plt.xlabel('ROC-AUC')
plt.axvline(0.5, color='red', linestyle='--', linewidth=0.8, label='Random baseline')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Auto-select best model & strategy ──
all_roc_cols = [c for c in grand_summary.columns if 'ROC-AUC' in c]
best_val, best_model_name, best_strategy = 0, "", ""

for _, row in grand_summary.iterrows():
    for col in all_roc_cols:
        val = row[col]
        if pd.notna(val) and val > best_val:
            best_val        = val
            best_model_name = row["Model"]
            best_strategy   = col.split("(")[-1].replace(")", "").strip()

print(f"✅ Best Model    : {best_model_name}")
print(f"✅ Best Strategy : {best_strategy}")
print(f"✅ Best ROC-AUC  : {best_val}")

## 16. Final Model Evaluation & Threshold Tuning

In [ ]:
# ── Retrieve best model and scaler based on winning strategy ──
strategy_map = {
    "Default"       : (X_train_scaled,    y_train,    scaler,    None),
    "SMOTE-Default" : (X_train_sm_scaled, y_train_sm, scaler_sm, None),
    "Under-Default" : (X_train_us_scaled, y_train_us, scaler_us, None),
    "Tuned-Original": (X_train_scaled,    y_train,    scaler,    tuned_orig_store),
    "Tuned-SMOTE"   : (X_train_sm_scaled, y_train_sm, scaler_sm, tuned_smote_store),
    "Tuned-Under"   : (X_train_us_scaled, y_train_us, scaler_us, tuned_under_store),
}

X_train_best, y_train_best, scaler_best, store = strategy_map[best_strategy]

if store and best_model_name in store:
    best_model = store[best_model_name]
else:
    # default strategy — refit
    best_model = all_model_instances[best_model_name]
    best_model.fit(X_train_best, y_train_best)

X_test_final = scaler_best.transform(X_test)
y_prob = best_model.predict_proba(X_test_final)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print(f"Model   : {best_model_name}")
print(f"Strategy: {best_strategy}")
print(f"ROC-AUC : {roc_auc_score(y_test, y_prob):.4f}")

In [ ]:
auc_score = roc_auc_score(y_test, y_prob)
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC = {auc_score:.4f}')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'ROC Curve — {best_model_name} ({best_strategy})', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Threshold tuning
best_t, best_f1_score = 0.5, 0
for t in np.arange(0.25, 0.65, 0.01):
    y_t = (y_prob >= t).astype(int)
    f   = f1_score(y_test, y_t)
    if f > best_f1_score:
        best_f1_score, best_t = f, t

print(f"Best threshold : {best_t:.2f}  →  Best F1: {best_f1_score:.4f}")
y_pred_tuned = (y_prob >= best_t).astype(int)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (preds, label) in zip(axes, [
    (y_pred,       "Default Threshold (0.50)"),
    (y_pred_tuned, f"Tuned Threshold ({best_t:.2f})")
]):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Churned','Churned'],
                yticklabels=['Not Churned','Churned'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix — {label}', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print("=== Default Threshold (0.50) ===")
print(classification_report(y_test, y_pred, target_names=['Not Churned','Churned']))
print(f"\n=== Tuned Threshold ({best_t:.2f}) ===")
print(classification_report(y_test, y_pred_tuned, target_names=['Not Churned','Churned']))

## 17. Overfitting / Underfitting Check — Learning Curve
- Train >> Validation → **Overfitting**
- Both low → **Underfitting**
- Both close and high → **Good fit ✅**

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_train_best, y_train_best,
    cv=5, scoring='roc_auc', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)
train_mean = np.mean(train_scores, axis=1)
test_mean  = np.mean(test_scores,  axis=1)
train_std  = np.std(train_scores,  axis=1)
test_std   = np.std(test_scores,   axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_mean, 'o-', color='steelblue', label='Training ROC-AUC')
plt.plot(train_sizes, test_mean,  'o-', color='coral',     label='Validation ROC-AUC')
plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.1, color='steelblue')
plt.fill_between(train_sizes, test_mean-test_std,   test_mean+test_std,   alpha=0.1, color='coral')
plt.xlabel('Training Size'); plt.ylabel('ROC-AUC Score')
plt.title('Learning Curve — Overfitting / Underfitting Check', fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()

gap = train_mean[-1] - test_mean[-1]
print(f"Train ROC-AUC : {train_mean[-1]:.4f}")
print(f"Val   ROC-AUC : {test_mean[-1]:.4f}")
print(f"Gap           : {gap:.4f}")
if gap > 0.1:   print("⚠️  Possible overfitting")
elif test_mean[-1] < 0.75: print("⚠️  Possible underfitting")
else:           print("✅ Good fit")

## 18. Feature Importances

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(
        best_model.feature_importances_, index=X.columns
    ).sort_values(ascending=False)
    importance.head(10).sort_values().plot(
        kind='barh', figsize=(10, 6), color='steelblue', edgecolor='white')
    plt.title(f'Top 10 Features — {best_model_name}', fontsize=13, fontweight='bold')
    plt.xlabel('Importance Score'); plt.tight_layout(); plt.show()
    print(importance.head(10))
else:
    print(f"{best_model_name} does not support feature_importances_ directly.")

## 19. Save Artifacts

In [ ]:
import pickle

pickle.dump(best_model,            open("churn_model.pkl",      "wb"))
pickle.dump(scaler_best,           open("scaler.pkl",            "wb"))
pickle.dump(X.columns.tolist(),    open("feature_columns.pkl",  "wb"))
pickle.dump(best_t,                open("best_threshold.pkl",   "wb"))

print("✅ Saved: churn_model.pkl, scaler.pkl, feature_columns.pkl, best_threshold.pkl")
print(f"   Best model    : {best_model_name}")
print(f"   Best strategy : {best_strategy}")
print(f"   Best ROC-AUC  : {best_val}")
print(f"   Best threshold: {best_t:.2f}")